In [1]:
from ugot import ugot
got = ugot.UGOT()
got.initialize("192.168.1.193")

got.load_models(["face_recognition"])

192.168.1.193:50051


True

In [8]:
got.get_face_recognition_total_info()

[['Stranger', 199.5, 217.0, 282, 235, 66270]]

In [ ]:

import cv2
import numpy as np
from IPython.display import display, Image, clear_output

got.open_camera()
try:
    while True:
        # Grab the latest camera frame as raw bytes
        frame = got.read_camera_data()

        # Decode the bytes into an OpenCV image array
        nparr = np.frombuffer(frame, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        # Get face recognition result: [[name, center_x, center_y, height, width, area], ...]
        faces = got.get_face_recognition_total_info()

        if faces:
            for face in faces:
                name, cx, cy, h, w, area = face
                cx, cy = int(cx), int(cy)
                half_w, half_h = int(w / 2), int(h / 2)

                # Compute bounding box corners
                x1, y1 = cx - half_w, cy - half_h
                x2, y2 = cx + half_w, cy + half_h

                # Draw bounding box and center dot
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.circle(img, (cx, cy), 4, (0, 0, 255), -1)

                # Label with name
                label = f"{name}  area: {area}px"
                cv2.putText(
                    img, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2
                )

        # Encode the annotated frame as JPEG and display it inline in the notebook
        _, jpeg = cv2.imencode(".jpg", img)
        clear_output(wait=True)
        display(Image(data=jpeg.tobytes()))

except KeyboardInterrupt:
    print("Done")
    got.mecanum_stop()